# Hybrid RAG for ERP — pełny przebieg

Korpus: **40 procedur / 185 kroków / 5 modułów**. Po ingeście graf ma
**141 węzłów Krok, 131 Stan, 185 krawędzi HAS_STEP**.

Sekcje 0–1 wykonujesz raz. Sekcje 2–8 przy każdej zmianie korpusu.

⚠️ **Po zmianie `.env` restartuj kernel.** `autoreload` przeładowuje kod,
ale nie wartości odczytane przy imporcie (`AGENT_MODEL`, `EMBEDDING_DIM`…).


In [1]:
%load_ext autoreload
%autoreload 2


---
## 0. Środowisko

Jednorazowo. `.env` musi zawierać: `LLM_MODEL`, `GRAPH_BUILDER_MODEL`,
`EMBEDDING_MODEL`, `EMBEDDING_DIM`, `GRAPH_DB_URL`, `GRAPH_DB_PASSWORD`.


In [2]:
!pip install -r requirements.txt
!pip install fastapi uvicorn requests



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from app.core import AGENT_MODEL, GRAPH_BUILDER_MODEL, EMBEDDING_MODEL, EMBEDDING_DIM

print(f'{AGENT_MODEL=}')
print(f'{GRAPH_BUILDER_MODEL=}')
print(f'{EMBEDDING_MODEL=}')
print(f'{EMBEDDING_DIM=}')


AGENT_MODEL='qwen3.5:4b'
GRAPH_BUILDER_MODEL='gpt-5.6-luna'
EMBEDDING_MODEL='bge-m3:latest'
EMBEDDING_DIM=1024


### Baza grafowa


In [4]:
!docker compose -f database/docker-compose.yml --env-file .env up -d


 Network erp-assistant-dbs_default Creating 
 Network erp-assistant-dbs_default Created 
 Container erp-assistant-graph-db Creating 
 Container erp-assistant-graph-db Created 
 Container erp-assistant-graph-db Starting 
 Container erp-assistant-graph-db Started 


### Modele

Przy Ollamie lokalnej pobierz je tutaj. Przy OpenWebUI albo OpenAI pomiń.


In [ ]:
!ollama pull {AGENT_MODEL}
!ollama pull {GRAPH_BUILDER_MODEL}
!ollama pull {EMBEDDING_MODEL}


---
## 1. Sanity check

Jeden import wyłapuje literówki, braki w `.env` i złe ścieżki.


In [2]:
import app.core, app.builder, app.schema, app.graph, app.plan
import app.ingest, app.assistant

from app.graph import PROMPTS_DIR, GRAPHS_DIR
from app.ingest import KNOWLEDGE_DIR

print('importy OK')
print(f'{PROMPTS_DIR}  istnieje={PROMPTS_DIR.exists()}')
print(f'{KNOWLEDGE_DIR}  istnieje={KNOWLEDGE_DIR.exists()}')


importy OK
C:\Users\BKZ\Desktop\Hybrid-RAG-For-ERP-Management\system  istnieje=True
C:\Users\BKZ\Desktop\Hybrid-RAG-For-ERP-Management\knowledge  istnieje=True


### Schema ma komplet pól


In [3]:
from app.schema import Procedure, ProcedureStep, AskAction

for pole in ('optional', 'requires', 'provides', 'why'):
    assert pole in ProcedureStep.model_fields, f'ProcedureStep bez pola {pole}'
assert 'goal' in Procedure.model_fields, 'Procedure bez pola goal'
print('schema_n OK — pola:', sorted(ProcedureStep.model_fields))


schema_n OK — pola: ['action', 'anchor', 'note', 'optional', 'provides', 'requires', 'text', 'why']


### Modele odpowiadają


In [4]:
from app.core import ChatModel, EmbeddingModel, AGENT_MODEL, EMBEDDING_MODEL, EMBEDDING_DIM

chat = ChatModel(model=AGENT_MODEL, system='Odpowiadasz jednym zdaniem.', memory=False)
print('czat:', chat.ask('Powiedz cokolwiek.', think=False)[:90])
chat.close()

resp = EmbeddingModel(EMBEDDING_MODEL).embed('test')
wymiar = len(resp.embeddings[0])
print(f'embedding: model={resp.model} wymiar={wymiar} EMBEDDING_DIM={EMBEDDING_DIM} '
      f'zgodne={wymiar == EMBEDDING_DIM}')


czat: Mogę opisać, jak zachodnie słońce maluje niebo w odcieniach pomarańczu i fioletu przed zmi
embedding: model=bge-m3:latest wymiar=1024 EMBEDDING_DIM=1024 zgodne=True


---
## 2. Walidacja korpusu PRZED ingestem

Trzy sprawdzenia offline — tanie, a wychwytują to, co inaczej wyjdzie
dopiero po kilkunastu minutach pracy modelu.


In [5]:
from app.ingest import load_knowledge, check_for_duplicates
from app.schema import Procedure

docs = load_knowledge()
check_for_duplicates(docs)

procedury = [d for d in docs if isinstance(d, Procedure)]
kroki = sum(len(p.steps) for p in procedury)
print(f'Dokumentów: {len(docs)}  procedur: {len(procedury)}  kroków: {kroki}')
print('Oczekiwane: 40 procedur, 185 kroków')


Dokumentów: 78  procedur: 40  kroków: 185
Oczekiwane: 40 procedur, 185 kroków


**Stany bez producenta to BŁĄD.** Braki na starcie procedury są normalne —
to zależności międzyproceduralne, które domyka planer.


In [6]:
daje = {s for p in procedury for st in p.steps for s in st.provides}
wym  = {s for p in procedury for st in p.steps for s in st.requires}

print('Stany bez producenta:', sorted(wym - daje) or 'brak')
print('Stanów łącznie:', len(daje))
print('Bez celu:', [p.id for p in procedury if not p.goal] or 'brak')


Stany bez producenta: brak
Stanów łącznie: 131
Bez celu: brak


### Akcje autopilota

`ask` = krok, przy którym autopilot zapyta użytkownika o wartość.


In [10]:
from collections import Counter

akcje = Counter(st.action.kind for p in procedury for st in p.steps if st.action)
print('Akcje:', dict(akcje))
print('Kroków z why:', sum(1 for p in procedury for st in p.steps if st.why))
print('Kroków opcjonalnych:', sum(1 for p in procedury for st in p.steps if st.optional))


Akcje: {'click': 88, 'select': 33, 'fill': 33}
Kroków z why: 0
Kroków opcjonalnych: 1


### Audyt anchorów — korpus kontra realne UI

W konsoli przeglądarki wklej poniższe, **przeklikaj wszystkie ekrany
i formularze** (w tym Dodaj pozycję i Nowa lokalizacja), potem wklej wynik:

```javascript
window.__a ??= new Set();
setInterval(() => document.querySelectorAll('[data-assistant-id]')
  .forEach(el => window.__a.add(el.getAttribute('data-assistant-id'))), 800);
// na koniec:  copy([...window.__a].sort().join('\n'))
```


In [ ]:
ANCHORY_UI = """

"""

ui = {l.strip() for l in ANCHORY_UI.split() if l.strip()}

if ui:
    z_korpusu = {st.anchor for p in procedury for st in p.steps if st.anchor}
    print('W korpusie, BRAK w UI:', sorted(z_korpusu - ui) or 'brak')
    print('W UI, nieużywane:', len(ui - z_korpusu))
else:
    print('Pomiń, jeśli nie zbierasz anchorów')


---
## 3. Ingest

`purge_database` jest **obowiązkowy** — nazwy relacji i parametrów zmieniły
się na angielskie, więc stare węzły nie pasują do nowych zapytań.


In [ ]:
from app import graph as graph

graph.initialize_graph_driver()
graph.purge_database(driver=graph.graph_driver)
print('baza wyczyszczona')


Poniższa komórka zatrzyma się na `input()`. Zanim naciśniesz ENTER, sprawdź
linię z licznikami — oczekiwane:

```
kroki=141 współdzielone=44 krawędzie=185 stany=131 procedury=40
```

Jeśli pojawi się `UWAGA: N procedur nie ma węzła` — LLM nie trzymał się
konwencji nazw. Przerwij i popraw prompt systemowy.


In [10]:
from app.core import EmbeddingModel, GRAPH_BUILDER_MODEL, EMBEDDING_MODEL, EMBEDDING_DIM
from app.ingest import ingest_llm

embed = EmbeddingModel(EMBEDDING_MODEL)
graph.initialize_knowledge_graph()
ingest_llm(driver=graph.graph_driver,
           model=GRAPH_BUILDER_MODEL,
           embed=embed,
           dim=EMBEDDING_DIM)


Ingest LLM: VALIDATING
Ingest LLM: LOADING KNOWLEDGE
Ingest LLM: STRINGING KNOWLEDGE
Ingest LLM: BUILDING

[done_reason=completed, prompt=38657 tok, out=84 tok]


🔧 read_classes({})

🔧 read_relationships({})

🔧 read_node_names({})

   ↳ BŁĄD: Nie dodano jeszcze żadnej klasy! Aby dodać klasę skorzystaj z 'define_class'

   ↳ BŁĄD: Nie dodano jeszcze żadnej relacji! Aby dodać relację skorzystaj z 'define_relation'

   ↳ BŁĄD: Nie dodano jeszcze żadnego noda! Aby dodać node skorzystaj z 'merge'


[done_reason=completed, prompt=38852 tok, out=339 tok]


🔧 define_class({'class_name': 'Procedura', 'parameters': {'tytul': 'Przykładowy tytuł', 'streszczenie': 'Przykładowe streszczenie', 'zapytania': ['przykładowe zapytanie'], 'warunki_wstepne': ['brak danych'], 'role': ['magazynier'], 'bledy': ['ERR_0000']}, 'parameters_to_embed': ['tytul', 'streszczenie', 'zapytania', 'warunki_wstepne']})

🔧 define_class({'class_name': 'Blad', 'parameters': {'kod': 'ERR_0000', 'opis': 'Opis błędu', 'przyczyny

Zapisywanie relacji: 100%|██████████| 622/622 [00:00<00:00, 840.41it/s]

Ingest LLM: SAVING
Ingest LLM: DONE! Możesz podglądać wyniki na: http://localhost:7474


---
## 4. Weryfikacja grafu


In [11]:
from collections import Counter
kg = graph.knowledge_graph

print('Klasy:  ', list(kg.classes.keys()))
print('Relacje:', list(kg.relations.keys()))
print('\nWęzły wg klas:   ', dict(Counter(n.c_name for n in kg.nodes.values())))
print('Węzły wg modułów:', dict(Counter(n.module for n in kg.nodes.values())))


Klasy:   ['Procedura', 'Blad', 'Pojecie', 'Krok', 'Stan']
Relacje: ['MA_BLAD', 'WYMAGA', 'DOTYCZY_POJECIA', 'HAS_STEP', 'REQUIRES', 'REQUIRES_STATE', 'PROVIDES_STATE', 'HAS_GOAL', 'RESOLVED_BY']

Węzły wg klas:    {'Procedura': 40, 'Blad': 23, 'Pojecie': 15, 'Krok': 140, 'Stan': 131}
Węzły wg modułów: {'inwentaryzacja': 29, 'lokalizacje': 32, 'magazyn': 121, 'sprzedaz': 80, 'zakupy': 87}


### Konwencja nazw węzłów


In [12]:
from app.plan import node_id_from_document_id

brak = [p.id for p in procedury if node_id_from_document_id(p.id) not in kg.nodes]
print('Procedury bez węzła:', brak or 'brak — konwencja zachowana')


Procedury bez węzła: brak — konwencja zachowana


### Walidacja korpusu w bazie

`unused_states` wskaże stany końcowe (`inw.zamknieta`, `fz.zaksiegowana`…)
— to **nie błąd**, tylko koniec łańcucha. Pozostałe kategorie mają być puste.


In [13]:
from app.plan import validate_corpus

problemy = validate_corpus(graph.graph_driver)

for kategoria, lista in problemy.items():
    print(f'\n{kategoria}  ({len(lista)})')
    for x in lista[:20]:
        print('   ', x)

if not problemy:
    print('Korpus w pełni spójny')


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `REQUIRES` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=25, offset=25>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 25, 'line': 2, 'column': 25}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            MATCH (a)-[:REQUIRES*1..10]->(a)\n            RETURN DISTINCT a.node_id AS problem'



unused_states  (50)
    inw.roznice-sprawdzone
    inw.zamknieta
    inw.postep-sprawdzony
    lok.pojemnosc
    lok.opis
    lok.zapisana
    lok.dezaktywowana
    lok.filtr-magazyn
    lok.aktywowana
    dokument.typ-pz
    dokument.kontrahent
    dokument.magazyn-docelowy
    dokument.zatwierdzony
    dokument.typ-wz
    dokument.typ-mm
    dokument.zatwierdzony-mm
    stany.przefiltrowane
    stany.filtr-kategoria
    stany.widok-przefiltrowany
    dok.lista-przefiltrowana


---
## 5. Testy planowania

Sedno systemu: czy trawersja daje kompletne i sensowne plany.


In [20]:
from app.plan import (build_plan, full_plan, plan_for_goal, validate_plan,
                      load_step_index, goal_states)

index = load_step_index(graph.graph_driver)
print('Kroków w indeksie:', len(index))


Kroków w indeksie: 141


**Łańcuch trzech procedur** — realizacja zamówienia sprzedaży wymaga
utworzenia i potwierdzenia zamówienia.


In [21]:
rows = full_plan(graph.graph_driver, 'proc_sprzedaz_realizacja_zamowienia', index=index)

for i, r in enumerate(rows, 1):
    print(f"  {i:2}. [{r['procedure'][:24]:24}] {r['text'][:48]}")


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `REQUIRES` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=25, offset=81>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 81, 'line': 3, 'column': 25}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH path = (start:SHARED {node_id: $node_id})\n                     -[:REQUIRES*0..3]->(proc)\n        WITH proc, max(length(path)) AS depth\n        ORDER BY depth DESC, proc.node_id\n        WITH collect(proc) AS procedures\n        UNWIND range(0, size(procedures) - 1) AS idx\n        WITH procedures[idx

   1. [(warunek wstępny)       ] Przejdź do Zamówienia sprzedaży w menu bocznym
   2. [(warunek wstępny)       ] Kliknij Nowe zamówienie
   3. [(warunek wstępny)       ] Ustaw oczekiwaną datę realizacji
   4. [(warunek wstępny)       ] W polu Magazyn wydania wybierz, z którego magazy
   5. [(warunek wstępny)       ] W polu Odbiorca wybierz klienta, który składa za
   6. [(warunek wstępny)       ] W stopce karty Pozycje kliknij Dodaj pozycję
   7. [(warunek wstępny)       ] Wybierz produkt z listy
   8. [(warunek wstępny)       ] Wpisz zamawianą ilość
   9. [(warunek wstępny)       ] Kliknij Zapisz zamówienie w stopce karty
  10. [(warunek wstępny)       ] Otwórz szkic zamówienia klikając jego numer na l
  11. [(warunek wstępny)       ] W stopce karty Pozycje kliknij Potwierdź zamówie
  12. [proc_sprzedaz_realizacja] Otwórz zamówienie w statusie Potwierdzone klikaj
  13. [proc_sprzedaz_realizacja] W stopce karty Pozycje kliknij Zrealizuj (utwórz
  14. [proc_sprzedaz_realizacja] W szkicu

**Ten sam cel z kontekstem** — użytkownik ma już potwierdzone zamówienie.


In [22]:
rows = full_plan(graph.graph_driver, 'proc_sprzedaz_realizacja_zamowienia',
                 initial_state={'zs.potwierdzone'}, index=index)

print(f'{len(rows)} kroków (bez kontekstu było ~13)')
for r in rows:
    print('  ', r['text'][:60])


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `REQUIRES` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=25, offset=81>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 81, 'line': 3, 'column': 25}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH path = (start:SHARED {node_id: $node_id})\n                     -[:REQUIRES*0..3]->(proc)\n        WITH proc, max(length(path)) AS depth\n        ORDER BY depth DESC, proc.node_id\n        WITH collect(proc) AS procedures\n        UNWIND range(0, size(procedures) - 1) AS idx\n        WITH procedures[idx

4 kroków (bez kontekstu było ~13)
   Przejdź do Zamówienia sprzedaży w menu bocznym
   Otwórz zamówienie w statusie Potwierdzone klikając jego nume
   W stopce karty Pozycje kliknij Zrealizuj (utwórz WZ)
   W szkicu WZ zweryfikuj ilości i zatwierdź dokument


**Rola kierownika** — MM dla magazyniera ma krok zmiany uprawnień, dla
kierownika nie. Stan `rola.kierownik` wstrzykuje `assistant_n` z `context.role`.


In [ ]:
for stan, opis in [(set(), 'magazynier'), ({'rola.kierownik'}, 'kierownik')]:
    rows = full_plan(graph.graph_driver, 'proc_magazyn_przesuniecie_mm',
                     initial_state=stan, index=index)
    rola = any('Kontekst uprawnień' in r['text'] for r in rows)
    print(f'{opis:12} {len(rows)} kroków, krok zmiany roli: {rola}')


**Zadanie złożone** — dwa cele w jednym planie. To jest mechanizm, dzięki
któremu „utwórz zamówienie i wystaw fakturę" nie kończy się na pierwszym.


In [23]:
jeden = plan_for_goal(['zz.zapisane'], index, preferred_module='zakupy')
dwa   = plan_for_goal(['zz.zapisane', 'fz.zaksiegowana'], index, preferred_module='zakupy')

print(f'jeden cel: {len(jeden)} kroków')
print(f'dwa cele:  {len(dwa)} kroków, bez duplikatów: {len({k["node_id"] for k in dwa})}')
for k in dwa:
    print(f"   [{k['module']:9}] {k['text'][:52]}")


jeden cel: 9 kroków
dwa cele:  17 kroków, bez duplikatów: 17
   [zakupy   ] Przejdź do Zamówienia zakupu w menu bocznym
   [zakupy   ] Kliknij Nowe zamówienie
   [zakupy   ] Ustaw oczekiwaną datę dostawy
   [zakupy   ] W polu Dostawca wybierz kontrahenta, u którego zamaw
   [zakupy   ] W polu Magazyn dostawy wybierz, dokąd ma trafić towa
   [zakupy   ] W stopce karty Pozycje kliknij Dodaj pozycję
   [zakupy   ] Wybierz produkt z listy
   [zakupy   ] Wpisz zamawianą ilość
   [zakupy   ] Kliknij Zapisz zamówienie w stopce karty
   [zakupy   ] Przejdź do Faktury zakupu w menu bocznym
   [zakupy   ] Kliknij Nowa faktura
   [zakupy   ] W sekcji Terminy ustaw datę wystawienia
   [zakupy   ] Jeśli faktura nie dotyczy żadnego zamówienia, wskaż 
   [zakupy   ] Wpisz numer faktury nadany przez dostawcę
   [zakupy   ] Ustaw termin płatności
   [zakupy   ] Kliknij Zapisz fakturę
   [zakupy   ] Otwórz zapisaną fakturę i kliknij Zaksięguj, gdy wsz


**Osiągalność wszystkich celów.**


In [ ]:
from app.plan import NoStepError

ok = True
for p in procedury:
    try:
        plan_for_goal(p.goal, index, preferred_module=p.module)
    except NoStepError as e:
        print('  NIEOSIĄGALNY:', p.id, str(e)[:70]); ok = False

print(f'Wszystkie {len(procedury)} celów osiągalne: {ok}')


**Test negatywny** — odwrócony plan MUSI zgłosić błędy. Gdyby zwrócił pustą
listę, walidacja nie działa.


In [ ]:
kroki_id = [r['step_id'] for r in build_plan(graph.graph_driver, 'proc_magazyn_przyjecie_pz')]

print('Poprawny :', validate_plan(graph.graph_driver, kroki_id) or 'bez zarzutu')
print('Odwrócony:', validate_plan(graph.graph_driver, list(reversed(kroki_id)))[:3])


---
## 6. Strojenie MIN_SCORE

Ustaw `ASSISTANT_MIN_SCORE` w `.env` **powyżej** najlepszego trafienia dla
pytań spoza korpusu i **poniżej** najgorszego dla sensownych.

⚠️ Po zmianie `.env` restartuj kernel.


In [3]:
graph.initialize_embed_model()

pytania = [
    ('W korpusie', 'jak przyjąć towar na magazyn'),
    ('W korpusie', 'jak wysłać zamówienie do dostawcy'),
    ('W korpusie', 'jak zamknąć inwentaryzację'),
    ('W korpusie', 'jak dodać nowy regał'),
    ('W korpusie', 'jak dodać produkt do kartoteki'),
    ('W korpusie', 'gdzie sprawdzę cenę ewidencyjną'),
    ('SPOZA',      'jaka jest stolica Francji'),
    ('SPOZA',      'jak ugotować makaron'),
    ('SPOZA',      'ile kosztuje licencja SAP'),
]

for etykieta, q in pytania:
    wektor = graph.embedding_model.embed(q).embeddings[0]
    wyniki = graph.KnowledgeGraph.search_semantic(graph.graph_driver, wektor, top_k=3)
    print(f'\n[{etykieta}] {q}')
    for r in wyniki:
        print(f"   {r['score']:.3f}  {r.get('klasa',''):10} {r['node_id']}")



[W korpusie] jak przyjąć towar na magazyn
   0.850  Procedura  proc_magazyn_przyjecie_pz
   0.790  Procedura  proc_magazyn_wydanie_wz
   0.790  Krok       krok_b29c9f147b9c

[W korpusie] jak wysłać zamówienie do dostawcy
   0.844  Procedura  proc_zakupy_wyslanie_zamowienia
   0.827  Procedura  proc_zakupy_utworzenie_zamowienia
   0.820  Krok       krok_028589381615

[W korpusie] jak zamknąć inwentaryzację
   0.854  Krok       krok_fb26a0f681b8
   0.833  Procedura  proc_inwentaryzacja_zamkniecie
   0.824  Blad       ERR_4003

[W korpusie] jak dodać nowy regał
   0.803  Krok       krok_201ad739b8f6
   0.784  Krok       krok_e87f824b3a0c
   0.784  Krok       krok_d258887ebad4

[W korpusie] jak dodać produkt do kartoteki
   0.831  Procedura  proc_magazyn_dodanie_produktu
   0.803  Procedura  proc_sprzedaz_dodanie_kontrahenta
   0.800  Procedura  proc_magazyn_wyszukanie_produktu

[W korpusie] gdzie sprawdzę cenę ewidencyjną
   0.879  Krok       krok_95ad0f21ede4
   0.808  Krok       krok_0

---
## 7. Warstwa asystenta (bez HTTP)

Łatwiej debugować niż przez serwer.


In [ ]:
from app.assistant import answer, recovery_plan, get_index

get_index(reload=True)

def zapytaj(q, ctx=None):
    o = answer(q, ctx)
    print(f'\n=== {q}   {ctx or ""}')
    print(f"   refused={o['refused']}  kroków={len(o['steps'])}  sources={o['sources']}")
    print(f"   {o['text'][:120]}")
    for s in o['steps'][:5]:
        akcja = s.get('action', {}).get('kind')
        print(f"      - {s['text'][:48]:50} {akcja or ''}")
    return o


### Pojedyncze procedury


In [ ]:
zapytaj('jak przyjąć towar na magazyn')
zapytaj('jak dodać nową lokalizację')
zapytaj('jak dodać produkt do kartoteki')
zapytaj('gdzie sprawdzę stan minimalny produktu')


### Łańcuchy międzyproceduralne


In [ ]:
zapytaj('jak wysłać towar do klienta z zamówienia')
zapytaj('jak zamknąć inwentaryzację')
zapytaj('jak zarejestrować fakturę od dostawcy')


### Zadanie złożone — dwie procedury w jednej odpowiedzi

To jest test wielozadaniowości. Plan ma pokryć OBA zadania.


In [ ]:
o = zapytaj('utwórz zamówienie zakupu i zarejestruj do niego fakturę')
print('\nProcedury źródłowe:', o['sources'])


### Kontekst UI skraca plan


In [ ]:
zapytaj('jak zrealizować zamówienie sprzedaży')

zapytaj('jak zrealizować zamówienie sprzedaży', {
    'route': '/sales-orders/so-1',
    'visibleActions': [{'id': 'btn.so-fulfil', 'disabled': False}],
})


### Rola z kontekstu usuwa krok zmiany uprawnień


In [ ]:
zapytaj('jak zrobić przesunięcie MM', {'role': 'magazynier'})
zapytaj('jak zrobić przesunięcie MM', {'role': 'kierownik'})


### Pojęcie, odmowa z podpowiedzią, pole na którym user utyka


In [ ]:
zapytaj('co to jest indeks produktu')
zapytaj('jaka jest stolica Francji')

zapytaj('nie wiem co tu wpisać', {
    'route': '/documents/new',
    'strugglingWith': 'field.counterparty',
    'form': {'fields': [{'id': 'field.counterparty', 'label': 'Dostawca',
                         'filled': False, 'invalid': True}]},
})


### Naprawa błędu — kod z bannera prowadzi do procedury naprawczej

To jest najmocniejszy moment demonstracji: autopilot klika, wyskakuje błąd,
asystent sam pokazuje, jak go naprawić.


In [ ]:
for kod in ('ERR-1004', 'ERR-4001', 'ERR-8002'):
    o = recovery_plan(kod, {'route': '/documents/d-1'})
    print(f"\n{kod}: refused={o['refused']} kroków={len(o['steps'])}")
    print('  ', o['text'][:90])
    for s in o['steps'][:3]:
        print('   -', s['text'][:60])


---
## 8. API

Serwer uruchom w **osobnym terminalu**:

```bash
uvicorn app.api_n:app --reload --port 8000
```


In [ ]:
import requests

print(requests.get('http://localhost:8000/assistant/health').json())


### Asercja kontraktu AssistantReply

Bramka przed podpięciem frontu — sprawdza kształt, nie treść.


In [ ]:
import requests

def sprawdz(q, ctx=None):
    o = requests.post('http://localhost:8000/assistant/ask',
                      json={'question': q, 'context': ctx}).json()

    assert set(o) == {'text', 'steps', 'sources', 'refused'}, o.keys()
    assert isinstance(o['text'], str) and isinstance(o['refused'], bool)
    assert isinstance(o['sources'], list)

    for s in o['steps']:
        assert set(s) <= {'text', 'anchor', 'action', 'note', 'why'}, s
        assert isinstance(s['text'], str) and s['text']
        if 'action' in s:
            a = s['action']
            assert a['kind'] in {'navigate','click','fill','select','ask'}, a
            assert a.get('anchor') or a['kind'] == 'navigate', a
            if a['kind'] == 'ask':
                assert a['inputType'] in {'text','number','date','select'}, a
                assert a.get('label'), a

    print(f"OK  refused={o['refused']}  kroków={len(o['steps'])}  {q}")
    return o

sprawdz('jak przyjąć towar na magazyn')
sprawdz('jak wysłać towar do klienta z zamówienia')
sprawdz('co to jest indeks produktu')
sprawdz('jaka jest stolica Francji')
sprawdz('jak zrealizować zamówienie', {'route': '/sales-orders/so-1'})


### Endpoint naprawczy


In [ ]:
o = requests.post('http://localhost:8000/assistant/recover',
                  json={'code': 'ERR-1004', 'context': {}, 'attempt': 1}).json()
print(f"kroków={len(o['steps'])}  {o['text'][:80]}")

# Limit prób: powyżej 2 serwer odmawia, żeby naprawa nie zapętliła się
o = requests.post('http://localhost:8000/assistant/recover',
                  json={'code': 'ERR-1004', 'context': {}, 'attempt': 9}).json()
print('proba 9 ->', o['refused'], '|', o['text'][:70])


### Pełna odpowiedź z akcjami autopilota


In [ ]:
o = sprawdz('jak przyjąć towar na magazyn')

print('\n' + o['text'] + '\n')
for i, s in enumerate(o['steps'], 1):
    print(f"{i}. {s['text']}")
    if s.get('why'):    print(f"      po co: {s['why']}")
    if s.get('action'): print(f"      akcja: {s['action']}")
    if s.get('note'):   print(f"      uwaga: {s['note']}")


### Po ponownym ingeście przeładuj bufor indeksu w działającym serwerze


In [ ]:
print(requests.post('http://localhost:8000/assistant/reload').json())


---
## 9. Front

Serwer ERP i front w osobnych terminalach:

```bash
npm install
npm run dev
```

Asystent w trybie atrapy (bez backendu Python) — `packages/web/.env.local`:

```
VITE_ASSISTANT_MOCK=1
```

Bez tej zmiennej front woła `/assistant/*`, które Vite proxuje na port 8000.


In [ ]:
!start http://localhost:5173


---
## 10. Kopie zapasowe grafu

`ingest_llm` zapisuje trzy razy: `preprocess` (po LLM), `poststeps`
(po dobudowaniu kroków) i `final` (po sync, z wektorami).


In [ ]:
from app.graph import GRAPHS_DIR, latest_graph

for katalog in ('preprocess', 'poststeps', 'final'):
    sciezka = GRAPHS_DIR / katalog
    if not sciezka.exists():
        continue
    print(f'\n{katalog}:')
    for f in sorted(sciezka.glob('*.json'), reverse=True)[:3]:
        print(f'  {f.stat().st_size/1024:8.1f} KB  {f.name}')


Wczytanie kopii — ratunek po nieudanym `sync()`, bez powtarzania pracy modelu.


In [ ]:
# graph.load_graph(latest_graph(GRAPHS_DIR / 'poststeps'))
# print(len(graph.knowledge_graph.nodes), 'węzłów wczytanych')
